In [7]:
import requests
import pandas as pd

Get all event_tickers for the Serie

In [8]:
EVENTS_API = "https://api.elections.kalshi.com/trade-api/v2/events" 
#SERIES_TICKER = "KXCPIYOY" #possible tickers for inflation: KXCPI, KXCPIYOY, KXLCPIMAXYOY, KXECONSTATCPIYOY, KXCPICOREYOY, KXECONSTATCPICORE (source: https://kalshi.com/category/economics/inflation)

def get_eventtickers_from_series(series_ticker):
    series_params= {
            "series_ticker": series_ticker,
            "status": "settled",
            "limit": 200 #max
        }
    response = requests.get(EVENTS_API, params=series_params)
    events_data = response.json()['events']
    events_df = pd.DataFrame(events_data)
    return events_df['event_ticker']

Get all the markets for an event

In [9]:

HISTORICAL_API = "https://api.elections.kalshi.com/trade-api/v2/historical/markets"
#EVENT_TICKER = "CPIYOY-23MAY" # KXCPIYOY-26FEB	OF KXCPI-26FEB

def get_markets_from_event(event_ticker):
    event_params= {
        "event_ticker": event_ticker,
        "status": "settled",
        "limit": 1000 #max
    }
    response = requests.get(HISTORICAL_API, params=event_params)
    market_data = response.json()['markets']
    market_data_df = pd.DataFrame(market_data)
    #display(market_data_df.head(3)) #we want: ticker, floor_strike, last_price_dollars, volume_fp, yes_ask_dollars

    return market_data_df



Run all code

In [14]:
event_tickers = get_eventtickers_from_series("KXCPIYOY") #must be all capitals
market_dfs = []
market_data_by_event = {}
for event_ticker in event_tickers:
    market_data = get_markets_from_event(event_ticker)
    market_dfs.append({
        "event_ticker": event_ticker,
        "market_dataframe": market_data
    })
    market_data_by_event[event_ticker] = market_data

market_dfs = pd.DataFrame(market_dfs)
# all_market_data = pd.concat(market_dfs["market_dataframe"].tolist(), ignore_index=True)
display(market_dfs.head())
market_dfs.to_excel("market_dfs.xlsx", index=False)
print(f"Saved market_dfs to market_dfs.xlsx")


,event_ticker,market_dataframe
0,KXCPIYOY-26FEB,Empty DataFrame Columns: [] Index: []
1,KXCPIYOY-26JAN,Empty DataFrame Columns: [] Index: []
2,KXCPIYOY-25DEC,Empty DataFrame Columns: [] Index: []
3,KXCPIYOY-25NOV,can_close_early close_time ...
4,KXCPIYOY-25OCT,can_close_early close_time ...


Saved market_dfs to market_dfs.xlsx


In [ ]:

import math

def score_event_markets(market_data):
    """Return two scoring digits for betting behavior and prediction consensus."""
    columns = [
        "expiration_value",
        "floor_strike",
        "last_price_dollars",
        "liquidity_dollars",
        "no_ask_dollars",
        "no_bid_dollars",
        "notional_value_dollars",
        "open_interest_fp",
        "previous_price_dollars",
        "previous_yes_ask_dollars",
        "previous_yes_bid_dollars",
        "price_ranges",
        "result",
        "settlement_value_dollars",
        "ticker",
        "title",
        "volume_24h_fp",
        "volume_fp",
        "yes_ask_dollars",
        "yes_ask_size_fp",
        "yes_bid_dollars",
        "yes_bid_size_fp",
    ]

    market_data = market_data.copy()
    market_data = market_data[[c for c in columns if c in market_data.columns]]

    if market_data.empty:
        return {
            "betting_behavior_score": 0.0,
            "prediction_score": 0.0,
            "predicted_side": None,
            "details": {},
        }

    total_volume = float(market_data["volume_fp"].sum()) if "volume_fp" in market_data else 0.0
    total_liquidity = float(market_data["liquidity_dollars"].sum()) if "liquidity_dollars" in market_data else 0.0
    avg_open_interest = float(market_data["open_interest_fp"].mean()) if "open_interest_fp" in market_data else 0.0
    avg_last_price = float(market_data["last_price_dollars"].mean()) if "last_price_dollars" in market_data else 0.0

    yes_price_cols = [c for c in ["yes_ask_dollars", "yes_bid_dollars"] if c in market_data]
    if yes_price_cols:
        yes_price_values = market_data[yes_price_cols].stack().astype(float)
        avg_yes_price = float(yes_price_values.mean()) if not yes_price_values.empty else avg_last_price
    else:
        avg_yes_price = avg_last_price

    # Betting behavior digit: overall market activity and depth.
    betting_behavior_score = (
        math.log1p(total_volume) * 0.4
        + math.log1p(total_liquidity + 1) * 0.4
        + math.log1p(abs(avg_open_interest) + 1) * 0.2
    )

    # Prediction digit: how strongly the market is pricing "yes" vs "no".
    prediction_score = min(max(avg_yes_price, 0.0), 1.0)
    predicted_side = "yes" if prediction_score >= 0.5 else "no"
    prediction_strength = abs(prediction_score - 0.5) * 2

    return {
        "betting_behavior_score": round(betting_behavior_score, 4),
        "prediction_score": round(prediction_score, 4),
        "predicted_side": predicted_side,
        "prediction_strength": round(prediction_strength, 4),
        "details": {
            "total_volume_fp": round(total_volume, 4),
            "total_liquidity_dollars": round(total_liquidity, 4),
            "avg_open_interest_fp": round(avg_open_interest, 4),
            "avg_yes_price": round(avg_yes_price, 4),
            "avg_last_price_dollars": round(avg_last_price, 4),
        },
    }

# Example usage for each event in the loaded data.
event_scores = {}
for event_ticker, market_data in market_data_by_event.items():
    event_scores[event_ticker] = score_event_markets(market_data)

scores_df = pd.DataFrame.from_dict(event_scores, orient="index")
display(scores_df)
print("Computed scoring digits for each event.")